# Enterprise Track 2 · A Lakeflow Pipeline with Expectations

**Goal:** run sessions 2-4 as a declarative pipeline: bronze (as received), silver (cleansed and typed), gold (lists and numbers), with every quality rule written as an **expectation** that the platform counts, enforces and records.

> ⚠️ **Mock data only.** This track runs on the agency workspace, which is approved for criminal justice information, but every table here holds the course's fictional records. Real tables come later, with the data owner's approval (see `enterprise/README.md`).

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and checks your settings.
STEP = 2

import importlib
import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import config, enterprise as ent
importlib.reload(config)                                         # picks up edits to lnl/config.py without a restart
N = ent.prepare(STEP, "Lakeflow pipeline")

## 1. What a declarative pipeline is
In session 5 you wrote each step and ran it. In a **Lakeflow pipeline** (open source as Spark Declarative Pipelines) you declare the tables you want and how each is computed from the others; the platform works out the order, runs only what changed, retries, and records everything in an event log.

| Layer | Table | Type | What it holds |
|---|---|---|---|
| Bronze | `bronze_offenders` | streaming table | every delivered row, as text, plus the file it came from |
| Silver | `silver_offenders` | streaming table | sessions 2-4 applied |
| Silver | `silver_quarantine` | streaming table | rows held back because an ID is broken |
| Gold | `gold_*` | materialized views | exception lists for the data owner, counts for reports |

A **streaming table** processes each new row once, so the next delivery costs only its own rows. **Auto Loader** (`cloudFiles`) is how the bronze table finds new files in the landing volume. A **materialized view** is recomputed from its inputs, so it's always consistent with them.

## 2. Expectations: three ways to enforce a rule
| Action | Decorator | What happens | Used here for |
|---|---|---|---|
| warn | `@dp.expect_all` | the row is kept; failures are counted in the event log | coded values not in a crosswalk, values that didn't convert, offense/violence conflicts, impossible birth dates |
| drop | `@dp.expect_all_or_drop` | the row stays out of the table; the drop is counted | broken TDCJ or SID numbers (they go to `silver_quarantine` for the data owner) |
| fail | `@dp.expect_or_fail` | the whole update stops | a delivery with a column the agreed layout doesn't have (Auto Loader puts it in `_rescued_data` instead of quietly adding it) |

These are the course's own rules: flag what a person must decide, hold back what can't be trusted, and stop when the input itself is wrong. Here's the pipeline's source:

In [ ]:
print((COURSE / ent.PIPELINE_FILE).read_text())

The pipeline doesn't copy the cleansing rules: it imports them from `lnl/spark_pipeline.py`, session 5's PySpark packaged as functions and tested to give the same numbers as the pandas sessions. One set of rules, three places it runs.

A missing column is the one layout problem the fail rule can't see: it just arrives as blanks. The warn rules and `validate_quality` catch that by its effect, a jump in blank or unmapped values.

## 3. Create and run the pipeline
`ensure_pipeline` creates the pipeline (or updates it if you run this again) with the Databricks SDK: serverless, in **development mode**, publishing to your schema. When you start an update from the pipeline's page, development mode skips automatic retries so a mistake shows up at once; updates started by the API (as this notebook does) or by a job are retried either way. Settings reach the code as **pipeline configuration**, never as hard-coded names.

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
pipeline_id = ent.ensure_pipeline(N)
print(f"Open it: {w.config.host}/pipelines/{pipeline_id}")

In [ ]:
first = ent.run_pipeline(pipeline_id)        # the first update builds every table: a few minutes

Open the link above while it runs: the graph shows each table, its row counts and its expectations as they're computed.

In [ ]:
display(ent.sql(f"""
    SELECT 'bronze_offenders' AS table_name, count(*) AS records FROM {N['fq']}.bronze_offenders
    UNION ALL SELECT 'silver_offenders', count(*) FROM {N['fq']}.silver_offenders
    UNION ALL SELECT 'silver_quarantine', count(*) FROM {N['fq']}.silver_quarantine"""))

In [ ]:
# One record, before and after: Superman, TDCJ #02381457 (stored as the BIGINT 2381457 in silver)
display(ent.sql(f"""
    SELECT b.tdcj_number AS as_received, b.unit_location, b.race, b.date_of_birth, b.restitution_owed,
           s.tdcj_number, s.unit_name, s.unit_code, s.race_code, s.date_of_birth AS dob, s.restitution_owed AS owed
    FROM {N['fq']}.bronze_offenders b
    JOIN {N['fq']}.silver_offenders s ON s.tdcj_number = CAST(b.tdcj_number AS BIGINT)
    WHERE b.tdcj_number = '02381457'"""))

## 4. What the expectations saw
Every update writes passed and failed counts for each warn and drop expectation to the **event log** (fail expectations stop the update instead of counting), which you query with the `event_log()` table function. Only the pipeline's run-as user (you) can read it by default. These are the same numbers session 2's exceptions list and session 4's conversion report produced, now recorded automatically on every run.

In [ ]:
display(ent.expectation_metrics(pipeline_id, first["update_id"]))

In [ ]:
display(spark.table(f"{N['fq']}.gold_quality_summary").orderBy("failing_records", ascending=False))

In [ ]:
display(ent.sql(f"""
    SELECT column, value_as_written, reason, count(*) AS records
    FROM {N['fq']}.gold_domain_exceptions
    GROUP BY ALL ORDER BY records DESC LIMIT 15"""))

## 5. The next delivery: incremental processing
A day later the source drops another file. The streaming tables process **only the new file**: the 100,000 rows from batch 1 aren't read again. Watch `_source_file`.

In [ ]:
ent.land_batch(N, 2)
second = ent.run_pipeline(pipeline_id)
display(ent.sql(f"SELECT _source_file, count(*) AS records FROM {N['fq']}.silver_offenders GROUP BY ALL ORDER BY 1"))

Two things to know:
- **A crosswalk change doesn't re-map rows already in silver.** Streaming tables process each row once. After a data steward changes reference data, run a **full refresh** (`ent.run_pipeline(pipeline_id, full_refresh=True)`) so every row is mapped with the new version.
- **Development vs production mode.** Development mode is for iterating. The job in notebook 04 runs the same pipeline the way production would: on a schedule, with retries and alerts.
- **An expectation on a materialized view means a full recompute.** `gold_people_by_unit` has one, so it's rebuilt on every update instead of incrementally. That's fine at this size; on a large table, put the check in `validate_quality` instead.

## Try it
- Add an expectation of your own to `enterprise/pipeline/hero_offenders.py`, for example `"custody level is present": "custody_level IS NOT NULL"` in `WARN_EXPECTATIONS` (in `lnl/spark_pipeline.py`), rerun the update, and find it in the event log.
- In the pipeline UI, click `silver_offenders` and open its **Data quality** tab.

**Next:** 03_governance controls who sees race and religion.